# 🔥 Qwen 3.8 27B Uncensored (huihui-ai) di Google Colab
Notebook ini menjalankan model **huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF** (IQ3_S 10.9 GB) secara **100% di GPU Nvidia T4 Colab**.

### ⚡ Cara Menjalankan (Cukup 1-Klik):
1. Pastikan GPU aktif: Menu **Runtime** -> **Change runtime type** -> Pilih **T4 GPU** -> **Save**.
2. Klik menu **Runtime** -> **Run all** (atau tekan tombol Play di Cell 1, lalu Cell 2).
3. Di bagian bawah Cell 2, klik link berwarna biru `https://xxxx.gradio.live` untuk langsung chatting!

### 📦 Langkah 1: Setup Engine & Download Model (Hanya butuh ~2 menit)

In [ ]:
# 1. Periksa GPU
import subprocess, time, os, urllib.request
try:
    gpu_info = subprocess.check_output(["nvidia-smi"]).decode()
    print("✅ GPU Terdeteksi:", gpu_info.split("\n")[8].strip())
except Exception:
    print("⚠️ PERINGATAN: GPU belum aktif! Klik menu Runtime -> Change runtime type -> Pilih T4 GPU -> Save, lalu jalankan ulang.")

# 2. Install aria2 (downloader cepat) & Engine Ollama
print("⏳ Menginstall Ollama & downloader aria2...")
!apt-get install -y -qq aria2 > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

# 3. Jalankan Ollama daemon di background
subprocess.Popen(["ollama", "serve"])
for _ in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags")
        print("✅ Engine Ollama aktif!")
        break
    except Exception:
        time.sleep(1)

# 4. Download Model Qwen 27B GGUF IQ3_S (10.9 GB)
model_url = "https://huggingface.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF/resolve/main/Huihui-Qwen3.8-27B-abliterated-GSQ-RCO-IQ3_S.gguf"
if not os.path.exists("/content/model.gguf") or os.path.getsize("/content/model.gguf") < 10000000000:
    print("\n⏳ Mendownload model 27B (~10.9 GB) dengan 16 koneksi paralel...")
    !aria2c -x 16 -s 16 -k 1M -c -o model.gguf "{model_url}"
else:
    print("✅ File model sudah ada di disk!")

# 5. Buat Modelfile dan muat ke GPU T4
with open("/content/Modelfile", "w") as f:
    f.write("""FROM /content/model.gguf
PARAMETER num_gpu 99
PARAMETER num_ctx 4096
PARAMETER temperature 0.7
PARAMETER top_p 0.9
""")

print("\n⚙️ Memuat model ke GPU T4...")
!ollama create qwen27b -f /content/Modelfile
print("\n🎉 MODEL 27B UNCENSORED SIAP DIGUNAKAN DI GPU!")

### 💬 Langkah 2: Buka Web Chat (Gradio UI)
Jalankan cell ini, lalu klik link **`Running on public URL: https://xxxx.gradio.live`** yang muncul di bawah.

In [ ]:
!pip install -q gradio requests

import gradio as gr
import requests, json

def format_history(message, history):
    messages = []
    if history:
        if isinstance(history[0], dict):
            messages = list(history)
        else:
            for user_msg, bot_msg in history:
                if user_msg: messages.append({"role": "user", "content": str(user_msg)})
                if bot_msg: messages.append({"role": "assistant", "content": str(bot_msg)})
    messages.append({"role": "user", "content": str(message)})
    return messages

def chat(message, history):
    payload = {
        "model": "qwen27b",
        "messages": format_history(message, history),
        "stream": True,
        "options": {
            "temperature": 0.7,
            "top_p": 0.9
        }
    }
    response = requests.post("http://localhost:11434/api/chat", json=payload, stream=True)
    partial_text = ""
    for line in response.iter_lines():
        if line:
            body = json.loads(line.decode("utf-8"))
            if "message" in body and "content" in body["message"]:
                partial_text += body["message"]["content"]
                yield partial_text

demo = gr.ChatInterface(
    fn=chat,
    title="🔥 Qwen 3.8 27B Uncensored (huihui-ai)",
    description="Model: Huihui-Qwen3.8-27B-abliterated | Engine: GPU T4 Colab | Bebas sensor, format rapi."
)

print("\n🚀 Membuka portal chat publik...")
demo.launch(share=True)